# Build a Sentiment Classifier with Word2Vec

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/nlp/04-embeddings/word2vec-classifier_exercise.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>


In the **Distributed Word Representations: Word2Vec** notebook, you trained a Word2Vec model to learn dense word vectors from text. In the **Build a Sentiment Analysis Classifier** lab, you used **CountVectorizer** and **TF-IDF** as sparse encoders and compared logistic regression with a linear SVM and `MLPClassifier`.

In this exercise, you replace CountVectorizer / TF-IDF with **Word2Vec** as the text encoder: tokenize each review, train Word2Vec on the training reviews, average word vectors into a document vector, then classify sentiment with the same sklearn models.

## Objective
In this lab, you will:
1. Preprocess review text and tokenize with Gensim
2. Train a Word2Vec model on training reviews only
3. Encode reviews as mean word vectors
4. Build a logistic regression model to predict sentiment
5. Train `LinearSVC` and `MLPClassifier` on the same Word2Vec features and compare all three models
6. **Bonus:** refactor encoding and classification into reusable functions (`encode_w2v`, `classify_lr` / `classify_svm` / `classify_mlp`)

## Scenario
You are an analyst for a marketing company that has just launched a new product suite of mobile devices. You have data from product reviews of one of these new products, the **TechWave X1**. For this lab you will use the text and star ratings from the reviews to predict customer sentiment.

## Dataset
- The Reviews data set (`reviews.csv` at `../data/reviews.csv`) is a small sample of reviews created for this lab


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/nlp/04-embeddings"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)

## Step 1: Import Libraries


In [ ]:
%pip install -qqq numpy pandas scikit-learn gensim


In [ ]:
# import numpy, pandas, gensim Word2Vec utilities, and the modules from scikit-learn
# that you need to run LogisticRegression, LinearSVC, and MLPClassifier.
import numpy as np
import pandas as pd

# Gensim for Word2Vec
from gensim.parsing.preprocessing import remove_stopwords
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec

# model building imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


In [ ]:
# code to avoid truncation of the output below
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


## Step 2: Load and Preprocess the Data


Use **Word2Vec** to predict the sentiment of reviews for the TechWave X1.

- First import the full data set from `reviews.csv` (`../data/reviews.csv`).
- Then convert the star rating into a binary outcome (0/1) to approximate sentiment analysis.


#### Read in the data and convert the `rating` feature to a numeric


In [ ]:
# INSERT YOUR CODE HERE


#### Create outcome variable from rating

You will convert the 5-star rating into a proxy for sentiment where star ratings:
- 1-3: Negative (0)
- 4-5: Positive (1)

Do this with a lambda function and `.apply()` and create a new sentiment column that will be used as the outcome in our model.


In [ ]:
# INSERT YOUR CODE HERE


In [ ]:
df_x1.head()


#### Create a training and test data set


In [ ]:
# INSERT YOUR CODE HERE


## Step 3: Tokenize the reviews

Word2Vec expects a list of token lists — one list of tokens per review. Use the helper below to lowercase, remove punctuation, and drop stop words.

Tokenize **both** the training and test reviews. Keep the raw `X_train` / `X_test` series for reference; you will also need lists of token lists for Word2Vec.


In [ ]:
def preprocess_text(text):
    """Clean and tokenize text using gensim's preprocessing utilities."""
    tokens = simple_preprocess(text)
    clean_text = remove_stopwords(' '.join(tokens))
    return clean_text.split()


In [ ]:
# INSERT YOUR CODE HERE
# Hint: processed_train = [preprocess_text(review) for review in X_train]


## Step 4: Train Word2Vec on training reviews only

Fit Word2Vec on **training tokens only** so test data does not leak into the embedding model.

See [Gensim's Word2Vec docs](https://radimrehurek.com/gensim/models/word2vec.html) if you need a reminder.

##### Hint!
```python
w2v_model = Word2Vec(
    sentences=processed_train,
    vector_size=100,
    window=5,
    min_count=1,
)
```


In [ ]:
# INSERT YOUR CODE HERE


## Step 5: Encode reviews as mean word vectors

Implement `review_to_vector(tokens, model)` that:
- looks up each token in `model.wv`
- skips out-of-vocabulary (OOV) words
- returns the **mean** of the known word vectors
- returns a zero vector if no tokens are in the vocabulary

Use it to build dense feature matrices for train and test splits.

##### Hint!
```python
def review_to_vector(tokens, model):
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)
```


In [ ]:
# INSERT YOUR CODE HERE


## Step 6: Build a logistic model to predict sentiment using Word2Vec features


Build a logistic regression model using the Word2Vec document vectors as features to predict sentiment.

Checkout the [scikit-learn logistic regression docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) if you need a reminder.

##### Hint!
```python
model = LogisticRegression()
```


In [ ]:
# INSERT YOUR CODE HERE


## Step 7: Compare three classifiers

Logistic regression is one linear classifier. A linear SVM ([`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)) finds a maximum-margin hyperplane in the same embedding space. [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) is the scikit-learn neural net from M1 — here it classifies dense Word2Vec features instead of sparse bag-of-words vectors.

Using your Word2Vec-encoded train/test features, fit and evaluate:

1. `LogisticRegression` (you can reuse the predictions from Step 6)
2. `LinearSVC`
3. `MLPClassifier` — set `random_state=42` and `max_iter=1000` so training is more likely to converge

Print accuracy and the confusion matrix for each model. Then write a short comparison: which model is strongest on this split, and does it still miss negative reviews?

##### Hint!
```python
model = LinearSVC()
model = MLPClassifier(random_state=42, max_iter=1000)
```


In [ ]:
# INSERT YOUR CODE HERE


## Refactor into named functions

Wrap the pipeline as reusable helpers and use them to run your three-model comparison.

```python
def preprocess_reviews(reviews):
    ...

def encode_w2v(processed_train, processed_test, vector_size=100, window=5, min_count=1):
    ...

def classify_lr(X_train, y_train, X_test, y_test):
    ...

def classify_svm(X_train, y_train, X_test, y_test):
    ...

def classify_mlp(X_train, y_train, X_test, y_test):
    ...
```


In [ ]:
# INSERT YOUR CODE HERE


## Conclusion

- Word2Vec produces **dense, low-dimensional** document vectors by averaging learned word embeddings. CountVectorizer and TF-IDF from the previous lab produce **sparse, high-dimensional** bag-of-words features.
- With a small training set, Word2Vec may underperform bag-of-words because it has fewer examples to learn good word vectors — but it captures semantic similarity between words that never co-occur in the same review.
- Record which of the three classifiers (`LogisticRegression`, `LinearSVC`, `MLPClassifier`) did best on your Word2Vec features, and whether it still missed negative reviews.
- Compare your Word2Vec results with CountVectorizer / TF-IDF from the **Build a Sentiment Analysis Classifier** lab. Which encoding worked better on this dataset, and why might that be?
